In [ ]:
!pip install -U ultralytics opencv-python transformers huggingface_hub

In [ ]:
from transformers import AutoModel
from PIL import Image
from ultralytics import YOLO
from huggingface_hub import hf_hub_download
import cv2
import requests
from yt_dlp import YoutubeDL
from google.colab.patches import cv2_imshow
from IPython.display import clear_output
import os

# Ejercicio 1: Gestión de clases en YOLO

Objetivo: Detectar y clasificar el tráfico en una calle.

## Qué hay que hacer:
- Buscar un modelo de detección de tráfico
- Comprobar qué clases es capaz de detectar el modelo (qué tipos de vehículos)
- Abrir la imagen dada para procesarla y aplica el modelo
- Mostrar una imagen con todos los vehículos detectados y su tipos
- Indica cuántos objetos se han detectado de cada clase

In [ ]:
import urllib.request

IMG_PATH = "https://raw.githubusercontent.com/jorgecs/apuntes/main/docs/ut5_ia_aplicada/4_yolo/img/street_photo.webp"
urllib.request.urlretrieve(IMG_PATH, "street_photo.webp")

In [ ]:
#TODO: Crea el modelo

#TODO: Predice las clases la imagen con el modelo

#TODO: Muestra la imagen con las boxes y la probabilidad

#TODO: Imprime cuántos elementos se han encontrado de cada clase

# Ejercicio 2: detección de personas con YOLO

Objetivo: usar un modelo YOLO para detectar personas en un vídeo y obtener el **máximo número de personas que se han visto a la vez**.

A continuación se descargará el vídeo con las personas a detectar y se recortará para que solamente ocupe lo necesario.

In [ ]:
url = "https://www.youtube.com/watch?v=vJG698U2Mvo"

ydl_opts = {
    "format": "bv*[vcodec^=avc1]+ba/b[ext=mp4]",
    "outtmpl": "/content/full_video.%(ext)s",
    "merge_output_format": "mp4",
}


with YoutubeDL(ydl_opts) as ydl:
    ydl.download([url])

VIDEO_PATH = "/content/selective_attention.mp4"


In [ ]:
!ffmpeg -ss 00:12 -to 01:02 -i /content/full_video.mp4 -c copy /content/selective_attention.mp4 -y

## Qué hay que hacer

1. Usa un modelo de Ultralytics YOLO para detección de objetos. Puedes usar yolov8n.pt (modelo base) o buscar un modelo en Hugging Face.
2. Si usas un modelo de Hugging Face, cárgalo con `hf_hub_download`. Si usas el modelo base de YOLO, con `ultralytics`.
3. Abre el vídeo para procesarlo.
4. Obtén el máximo de personas visibles en el vídeo.

IMPORTANTE: Hay que filtrar la clase `person`

In [ ]:
# TODO: Implementa el ejercicio 2 aquí

# Ejercicio 3: Conteo Real con Tracking

Objetivo: ¿Cuántas personas **distintas** han pasado por delante de la cámara en total?

En el ejercicio anterior contábamos el máximo de personas por frame. Pero si una persona sale y entra o si queremos saber el total acumulado, necesitamos **Tracking**.

El **tracking** añade un identificador (ID) a cada objeto para seguirlo en el tiempo, así podemos saber qué bounding box corresponde a qué persona en todo momento y contar cuántos ID únicos han aparecido.

El problema que puede tener es que si YOLO pierde de vista un objeto y lo vuelve a ver, puede asignarle un nuevo ID.

### Ejemplo de tracking

```python
from ultralytics import YOLO
import cv2

model = YOLO("yolov8n.pt")

#Abrimos el video con OpenCV para iterar frame a frame, como siempre
cap = cv2.VideoCapture("video.mp4")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    #En lugar de usar predict, usamos track, que nos devuelve un resultado similar pero con información de tracking (IDs asignados a cada objeto)
    results = model.track(frame, persist=True)[0]  #El parámetro persist=True hace que el modelo intente mantener el mismo ID para cada objeto a lo largo de los frames
    
    detecciones = results.boxes

    for box in detecciones:
        track_id = int(box.id[0])  #A cada detección se le asigna un ID por el tracker, que se puede acceder con box.id[0]

cap.release()
```

## Qué hay que hacer
1. Utiliza `model.track(..., persist=True)` en lugar de `.predict()`.
2. Los resultados ahora tendrán un campo `id` asociado (tracking ID).
3. Filtra la clase person si el modelo detecta más objetos.
4. Cuenta los IDs únicos (puedes usar una estructura tipo `set()` para evitar duplicados o comprobar si ya está almacenado).
5. Al final del vídeo, muestra el número total de personas únicas detectadas.

In [ ]:
# TODO: Implementa el tracking aquí usando el mismo vídeo y modelo que en el ejercicio anterior